### Music Clusterer

This notebook walks through data ingestion of audio + lyrics for the spotify dataset and provides clusters from a combination

In [ ]:
# Load dataset and configurations
import sys
import os
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.datasets import fetch_openml
from sklearn import metrics
import pandas as pd

# Get df and features from data_ingestion
sys.path.append("../Functions")
sys.path.append("../ThirdParty")
from data_ingestion import get_feature_data
X = get_feature_data()
#X = X.iloc[:500]  # use first 500 rows for development

In [ ]:
# Load cached lyrics and create SBERT embeddings (run once; cached to disk)
from genius import Genius, load_lyrics, save_lyrics
from data_ingestion import data_dir
CACHE_PATH = os.path.join(data_dir, "lyrics_cache.pkl")
EMBEDDINGS_PATH = os.path.join(data_dir, "lyric_embeddings.pkl")
client = Genius(os.environ["GENIUS_ACCESS_TOKEN"])  # token required by constructor; embed_text itself is offline
    
if os.path.exists(EMBEDDINGS_PATH):
    embeddings = load_lyrics(EMBEDDINGS_PATH)
    print(f"Loaded {len(embeddings)} embeddings from disk")
else:
    cache = load_lyrics(CACHE_PATH)
    embeddings = {}
    for key, lyrics in cache.items():
        if lyrics is None:
            continue
        try:
            embeddings[key] = client.embed_text(lyrics)
        except Exception as e:
            print(f"Embed failed for {key}: {e}")
    save_lyrics(embeddings, EMBEDDINGS_PATH)
    print(f"Embedded {len(embeddings)} of {len(cache)} cached songs; saved to {EMBEDDINGS_PATH}")

print(X.columns.tolist())

In [ ]:
# Align X with embeddings.
# X carries audio features only; track names + artists live in a separate metadata
# DataFrame that comes out of data_ingestion in the same row order as X.
import numpy as np
sys.path.append("../Functions")
from data_ingestion import get_track_data   # adjust name if yours differs

tracks = get_track_data()
assert len(tracks) == len(X), f"tracks ({len(tracks)}) and X ({len(X)}) must align row-for-row"

mask = tracks.apply(lambda r: (r["track_name"], r["artists"]) in embeddings, axis=1).values

X_aligned = X[mask].reset_index(drop=True)
tracks_aligned = tracks[mask].reset_index(drop=True)
keys = list(zip(tracks_aligned["track_name"], tracks_aligned["artists"]))
lyric_matrix = np.stack([embeddings[k] for k in keys]).astype(np.float32)

print(f"X:              {len(X)} songs")
print(f"embeddings:     {len(embeddings)} songs")
print(f"X_aligned:      {len(X_aligned)} songs (intersection)")
print(f"lyric_matrix:   {lyric_matrix.shape}")

In [ ]:
import hdbscan
from umap import UMAP

# 384D is too high for density clustering — UMAP-reduce first.
# n_components=15 keeps enough structure for HDBSCAN; min_dist=0.0 favors tight groups.
reducer = UMAP(n_components=15, n_neighbors=30, min_dist=0.0, metric="cosine", random_state=42)
lyric_umap = reducer.fit_transform(lyric_matrix)

lyric_clusterer = hdbscan.HDBSCAN(
    min_cluster_size=50,        # smallest group worth calling a cluster
    min_samples=1,             # how conservative to be about noise; lower = more clusters
    metric="euclidean",        # safe default for L2-normalized vectors
)
lyric_labels = lyric_clusterer.fit_predict(lyric_umap)

n_lyric_clusters = len(set(lyric_labels)) - (1 if -1 in lyric_labels else 0)
print(f"lyric clusters: {n_lyric_clusters}")
print(f"noise points:   {(lyric_labels == -1).sum()} ({(lyric_labels == -1).mean():.1%})")

In [ ]:
# Inspect lyric clusters: list a few songs from each
from collections import defaultdict

lyric_clusters = defaultdict(list)
for key, label in zip(keys, lyric_labels):
    lyric_clusters[label].append(key)

for cluster_id, songs in sorted(lyric_clusters.items()):
    print(f"\nCluster {cluster_id} ({len(songs)} songs):")
    for track, artist in songs[:5]:
        print(f"  {track} — {artist}")

In [ ]:
# Cosine similarity heatmap for the first N songs.
# Each cell (i, j) is the cosine similarity between songs i and j's lyric embeddings.
# Bright = lyrically similar, dark = lyrically different. The diagonal is always 1.0
# (a song matches itself). Off-diagonal bright blocks are early evidence of clusters.
# Because lyric_matrix is L2-normalized, the dot product == cosine similarity.
import plotly.express as px

size = 15
sample = lyric_matrix[:size]
sim_sample = sample @ sample.T

tick_labels = [f"{t[:20]} — {a[:15]}" for t, a in keys[:size]]
fig = px.imshow(
    sim_sample,
    x=tick_labels, y=tick_labels,
    color_continuous_scale="Viridis",
    aspect="auto",
    title=f"Pairwise lyric cosine similarity — first {size} songs",
)
fig.show()

In [ ]:
import numpy as np
from feature_correlation import *
# Find correlation between audio features
corr = feature_correlation(X)
plot_correlation_matrix(corr)

# Finding the inverse covariance of the scaled features  
VI = inv_cov_matrix(X)

In [ ]:
from umap import UMAP

# Using the covariance matrix to compute the Mahalanobis distance between points
reducer = UMAP(n_components=3,
               n_neighbors=100,
               min_dist=0.1,
               metric="mahalanobis",
               metric_kwds={"VI": VI},
               random_state=42)

audio_matrix = feature_scaling(X)
X_umap = reducer.fit_transform(audio_matrix)
X_umap.shape

In [ ]:
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "notebook"

fig = px.scatter_3d(
    x=X_umap[:, 0],
    y=X_umap[:, 1],
    z=X_umap[:, 2],
    #color=y_named
)

fig.show()

In [ ]:
# Cluster audio features with HDBSCAN, on the same songs as the lyric side
from feature_correlation import feature_scaling

audio_features_aligned = feature_scaling(X_aligned)

audio_clusterer = hdbscan.HDBSCAN(
    min_cluster_size=10,
    min_samples=5,
    metric="euclidean",
)
audio_labels = audio_clusterer.fit_predict(audio_features_aligned)

n_audio_clusters = len(set(audio_labels)) - (1 if -1 in audio_labels else 0)
print(f"audio clusters: {n_audio_clusters}")
print(f"noise points:   {(audio_labels == -1).sum()} ({(audio_labels == -1).mean():.1%})")

In [ ]:
import pandas as pd
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

contingency = pd.crosstab(
    pd.Series(audio_labels, name="audio"),
    pd.Series(lyric_labels, name="lyric"),
)

fig = px.imshow(
    contingency.values,
    x=[f"L{c}" for c in contingency.columns],
    y=[f"A{r}" for r in contingency.index],
    color_continuous_scale="Viridis",
    aspect="auto",
    labels={"x": "lyric cluster", "y": "audio cluster", "color": "count"},
    title="Audio × lyric cluster contingency",
)
fig.show()

# Numerical agreement — drop noise points on either side, otherwise -1==-1 inflates the score
both = (audio_labels != -1) & (lyric_labels != -1)
ari = adjusted_rand_score(audio_labels[both], lyric_labels[both])
nmi = normalized_mutual_info_score(audio_labels[both], lyric_labels[both])
print(f"Adjusted Rand Index:    {ari:.3f}   (0 = random, 1 = identical)")
print(f"Normalized Mutual Info: {nmi:.3f}   (0 = independent, 1 = perfectly predictable)")

In [ ]:
X_compare = X_aligned.copy()
X_compare["track_name"] = tracks_aligned["track_name"].values   # ← these two lines
X_compare["artists"] = tracks_aligned["artists"].values         # ← are the fix
X_compare["audio_cluster"] = audio_labels
X_compare["lyric_cluster"] = lyric_labels

for (a, l), group in X_compare.groupby(["audio_cluster", "lyric_cluster"]):
    if a == -1 or l == -1 or len(group) < 3:
        continue
    print(f"\nAudio cluster {a} ∩ Lyric cluster {l} — {len(group)} songs")
    for _, row in group.head(3).iterrows():
        print(f"  {row['track_name']} — {row['artists']}")

In [ ]:
import pandas as pd
import plotly.express as px
from umap import UMAP

# 2D layouts for plotting (separate from the 15D layout used for clustering)
audio_2d = UMAP(n_components=2, n_neighbors=30, min_dist=0.1,
                random_state=42).fit_transform(audio_features_aligned)
lyric_2d = UMAP(n_components=2, n_neighbors=30, min_dist=0.1,
                metric="cosine", random_state=42).fit_transform(lyric_matrix)

# Convert cluster labels to strings so plotly treats them as categorical (discrete colors)
plot_df = pd.DataFrame({
    "audio_x": audio_2d[:, 0],
    "audio_y": audio_2d[:, 1],
    "lyric_x": lyric_2d[:, 0],
    "lyric_y": lyric_2d[:, 1],
    "audio_cluster": [f"A{l}" if l != -1 else "noise" for l in audio_labels],
    "lyric_cluster": [f"L{l}" if l != -1 else "noise" for l in lyric_labels],
    "track":  tracks_aligned["track_name"].values,
    "artist": tracks_aligned["artists"].values,
})